# Autoencoder GRU, Mirai

Autoencoder secuencial que reconstruye ventanas de W = 3 registros consecutivos de una misma dirección IP de origen. La dirección IP de origen (`src_ip`) no se usa como entrada, porque identifica a los equipos atacantes.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

In [2]:
DATA_PATH = Path("../../data/flows.csv")
df = pd.read_csv(DATA_PATH)

# Ventanas por IP de origen, ordenadas por flow_start. src_ip solo sirve para agrupar, no es una entrada
df['group'] = df['src_ip']
df['order'] = df['flow_start']
df['is_attack'] = df['label']
X = pd.get_dummies(df.drop(columns=['class_name', 'flow_start', 'label', 'src_ip', 'group', 'order', 'is_attack']), columns=['state'])
meta = df[['group', 'order', 'is_attack']]
print(X.shape)

(1006776, 15)


In [3]:
def build_windows(X, meta, W, stride=1):
    meta = meta.copy()
    meta['pos'] = meta.groupby('group').cumcount()
    meta['grp_size'] = meta.groupby('group')['group'].transform('size')
    order_idx = meta.sort_values(['group', 'order']).index
    X = X.loc[order_idx].reset_index(drop=True)
    meta = meta.loc[order_idx].reset_index(drop=True)

    train_mask = (meta['pos'] < (0.8 * meta['grp_size'])).to_numpy()
    scaler = MinMaxScaler().fit(X.to_numpy()[train_mask])
    Xs = scaler.transform(X.to_numpy()).astype('float32')
    attack_arr = meta['is_attack'].to_numpy()
    group_arr = meta['group'].to_numpy()

    train_windows, test_windows, test_labels = [], [], []
    for gid in pd.unique(group_arr):
        sel = np.where(group_arr == gid)[0]
        is_train = train_mask[sel]
        train_idxs = sel[is_train]
        test_idxs = sel[~is_train]

        n_tr = len(train_idxs) - W + 1
        if n_tr > 0:
            attack_tr = attack_arr[train_idxs]
            cums = np.concatenate([[0], np.cumsum(attack_tr)])
            window_sums = cums[W:] - cums[:-W]
            for i in np.where(window_sums == 0)[0][::stride]:
                train_windows.append(Xs[train_idxs[i:i + W]])

        n_te = len(test_idxs) - W + 1
        if n_te > 0:
            attack_te = attack_arr[test_idxs]
            cums = np.concatenate([[0], np.cumsum(attack_te)])
            window_sums = cums[W:] - cums[:-W]
            for i in range(0, n_te, stride):
                test_windows.append(Xs[test_idxs[i:i + W]])
                test_labels.append(int(window_sums[i] > 0))

    X_train_w = np.array(train_windows) if train_windows else np.empty((0, W, X.shape[1]))
    X_test_w = np.array(test_windows) if test_windows else np.empty((0, W, X.shape[1]))
    y_test_w = np.array(test_labels)
    return X_train_w, X_test_w, y_test_w


W = 3
STRIDE = 1
X_train_w, X_test_w, y_test_w = build_windows(X, meta, W, stride=STRIDE)
print(f"Ventanas train: {X_train_w.shape}, ventanas test: {X_test_w.shape}, positivos test: {y_test_w.sum()}/{len(y_test_w)}")

Ventanas train: (6119, 3, 15), ventanas test: (201117, 3, 15), positivos test: 199837/201117


In [4]:
input_dim = X_train_w.shape[2]
bottleneck = 16

inputs = keras.Input(shape=(W, input_dim))
x = layers.GRU(32, return_sequences=True)(inputs)
x = layers.GRU(bottleneck, return_sequences=False)(x)
x = layers.RepeatVector(W)(x)
x = layers.GRU(bottleneck, return_sequences=True)(x)
x = layers.GRU(32, return_sequences=True)(x)
outputs = layers.TimeDistributed(layers.Dense(input_dim, activation='sigmoid'))(x)

autoencoder = keras.Model(inputs, outputs)
autoencoder.compile(optimizer='adam', loss='mse')

history = autoencoder.fit(
    X_train_w, X_train_w,
    epochs=100, batch_size=64, validation_split=0.1,
    callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)],
    verbose=0,
)
print(f"Epocas entrenadas: {len(history.history['loss'])}, val_loss final: {history.history['val_loss'][-1]:.5f}")

Epocas entrenadas: 100, val_loss final: 0.00029


In [5]:
def reconstruction_error(X):
    return np.mean(np.square(X - autoencoder.predict(X, verbose=0)), axis=(1, 2))

threshold = np.percentile(reconstruction_error(X_train_w), 95)
err_test = reconstruction_error(X_test_w)
y_pred = (err_test > threshold).astype(int)

print(classification_report(y_test_w, y_pred, target_names=['normal', 'ataque']))
print(confusion_matrix(y_test_w, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test_w, err_test):.4f}")

              precision    recall  f1-score   support

      normal       0.01      0.98      0.01      1280
      ataque       1.00      0.04      0.08    199837

    accuracy                           0.05    201117
   macro avg       0.50      0.51      0.05    201117
weighted avg       0.99      0.05      0.08    201117

[[  1250     30]
 [191349   8488]]
ROC-AUC: 0.9026
